In [0]:
%python
       
import pandas as pd

# ─────────────────────────────────────────────
# Avaliação de Qualidade de Dados — silver.aerodromos
# ─────────────────────────────────────────────

TABLE = "voebem.silver.aerodromos"

# 1) Schema da tabela
print("═" * 60)
print(f"1) SCHEMA — {TABLE}")
print("═" * 60)
schema_df = spark.sql(f"DESCRIBE TABLE {TABLE}").toPandas()
display(schema_df)

cols = [row.col_name for row in spark.sql(f"DESCRIBE TABLE {TABLE}").collect() if row.col_name and not row.col_name.startswith("#")]
print(f"\nTotal de colunas: {len(cols)}")

# 2) Contagem total de linhas
total_rows = spark.sql(f"SELECT COUNT(*) AS total FROM {TABLE}").collect()[0]["total"]
print("\n" + "═" * 60)
print(f"2) VOLUME — {total_rows:,} linhas")
print("═" * 60)

# 3) Completude — nulos e vazios por coluna
print("\n" + "═" * 60)
print("3) COMPLETUDE — nulos / vazios por coluna")
print("═" * 60)
null_exprs = [f"SUM(CASE WHEN `{c}` IS NULL OR CAST(`{c}` AS STRING) = '' THEN 1 ELSE 0 END) AS `{c}`" for c in cols]
null_sql = f"SELECT {', '.join(null_exprs)} FROM {TABLE}"
null_df = spark.sql(null_sql).toPandas().T.reset_index()
null_df.columns = ["coluna", "nulos_ou_vazios"]
null_df["perc_nulo"] = (null_df["nulos_ou_vazios"] / total_rows * 100).round(2)
null_df = null_df.sort_values("perc_nulo", ascending=False)
display(null_df)

# 4) Unicidade — duplicatas em colunas candidatas a chave
print("\n" + "═" * 60)
print("4) UNICIDADE — duplicatas em colunas candidatas a chave")
print("═" * 60)
key_candidates = [c for c in cols if any(k in c.lower() for k in ["id", "codigo", "cod", "icao", "iata"])]
if key_candidates:
    dup_results = []
    for kc in key_candidates:
        dup_count = spark.sql(f"""
            SELECT (COUNT(*) - COUNT(DISTINCT `{kc}`)) AS dup_count
            FROM {TABLE}
            WHERE `{kc}` IS NOT NULL
        """).collect()[0]["dup_count"]
        if dup_count > 0:
            dup_results.append((kc, dup_count))
            print(f"  ⚠️  Coluna '{kc}': {dup_count} valores duplicados")
        else:
            print(f"  ✅  Coluna '{kc}': sem duplicatas")
else:
    print("  Nenhuma coluna candidata a chave identificada (id, codigo, icao, iata...)")

# 5) Cardinalidade — valores distintos por coluna
print("\n" + "═" * 60)
print("5) CARDINALIDADE — valores distintos por coluna")
print("═" * 60)
card_exprs = [f"COUNT(DISTINCT `{c}`) AS `{c}`" for c in cols]
card_sql = f"SELECT {', '.join(card_exprs)} FROM {TABLE}"
card_df = spark.sql(card_sql).toPandas().T.reset_index()
card_df.columns = ["coluna", "valores_distintos"]
card_df["perc_distinto"] = (card_df["valores_distintos"] / total_rows * 100).round(2)
display(card_df)

# 6) Estatísticas básicas de colunas numéricas
print("\n" + "═" * 60)
print("6) ESTATÍSTICAS — colunas numéricas")
print("═" * 60)
numeric_cols = [row.col_name for row in spark.sql(f"DESCRIBE TABLE {TABLE}").collect()
                if row.col_name and not row.col_name.startswith("#")
                and any(t in (row.data_type or "").lower() for t in ["int", "double", "float", "decimal", "long", "bigint", "short"])]
if numeric_cols:
    summary_df = spark.sql(f"SELECT {', '.join(numeric_cols)} FROM {TABLE}").summary("min", "max", "mean", "stddev", "50%").toPandas()
    display(summary_df)
else:
    print("  Nenhuma coluna numérica encontrada.")

# 7) Amostra de dados
print("\n" + "═" * 60)
print("7) AMOSTRA — 10 linhas")
print("═" * 60)
display(spark.sql(f"SELECT * FROM {TABLE} LIMIT 10"))

# ─────────────────────────────────────────────
# 8) RESUMO DE RECOMENDAÇÕES DE QUALIDADE
# ─────────────────────────────────────────────
print("\n" + "═" * 60)
print("8) RECOMENDAÇÕES DE TRATAMENTO DE QUALIDADE")
print("═" * 60)

recommendations = []

# Completude
high_null = null_df[null_df["perc_nulo"] > 5]
for _, row in high_null.iterrows():
    if row["perc_nulo"] == 100.0:
        recommendations.append(f"🔴 `{row['coluna']}`: 100% nula — remover a coluna ou investigar origem no bronze.")
    elif row["perc_nulo"] > 50:
        recommendations.append(f"🟠 `{row['coluna']}`: {row['perc_nulo']}% nulo — avaliar se é opcional ou há falha no ETL.")
    else:
        recommendations.append(f"🟡 `{row['coluna']}`: {row['perc_nulo']}% nulo — aplicar imputação ou regra de default.")

# Unicidade
if 'dup_results' in dir() and dup_results:
    for kc, dc in dup_results:
        recommendations.append(f"🔴 `{kc}`: {dc} duplicatas — aplicar deduplicação (keep first/last por timestamp).")

# Cardinalidade (colunas com baixa cardinalidade podem ser categóricas)
low_card = card_df[(card_df["valores_distintos"] < 20) & (card_df["valores_distintos"] > 1)]
for _, row in low_card.iterrows():
    recommendations.append(f"🔵 `{row['coluna']}`: {row['valores_distintos']} valores distintos — validar domínio categórico (enum/lookup).")

# Colunas com alta cardinalidade (potencial PII ou chave)
high_card = card_df[card_df["perc_distinto"] > 95]
for _, row in high_card.iterrows():
    recommendations.append(f"🟣 `{row['coluna']}`: {row['perc_distinto']}% de valores distintos — candidata a chave primária ou PII.")

if not recommendations:
    recommendations.append("✅ Nenhum problema de qualidade evidente detectado.")

for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec}")

print("\n" + "─" * 60)
print("Próximos passos sugeridos:")
print("  • Implementar CHECK constraints no Delta table")
print("  • Adicionar expectation rules no pipeline (great expectations / Delta Live Tables)")
print("  • Criar tags de classificação no Unity Catalog para colunas sensíveis")
print("  • Configurar alertas de qualidade via system.table or DQX")
print("─" * 60)